In [3]:
from google.colab import files
uploaded = files.upload()

Saving sales_data_raw_for_task1.csv to sales_data_raw_for_task1 (2).csv


In [4]:

import pandas as pd
from google.colab import files

# STEP 1: Read the uploaded dataset
filename = next(iter(uploaded))
df = pd.read_csv(filename)

# STEP 2: Inspect the original dataset
print("===== ORIGINAL DATASET =====")
print(df.head())

print("\nDataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

# STEP 3: Clean column names
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# STEP 4: Remove duplicate rows
df = df.drop_duplicates().copy()

# STEP 5: Clean text columns
for column in df.select_dtypes(include="object").columns:
    df[column] = df[column].apply(
        lambda value: value.strip()
        if isinstance(value, str) else value
    )
    df[column] = df[column].replace("", pd.NA)

# STEP 6: Convert the date column
if "date" in df.columns:
    df["date"] = pd.to_datetime(
        df["date"], errors="coerce"
    )

# STEP 7: Convert numeric columns
for column in ["quantity", "unit_price", "customer_age"]:
    if column in df.columns:
        df[column] = pd.to_numeric(
            df[column], errors="coerce"
        )

# STEP 8: Fill missing numeric values with the median
for column in df.select_dtypes(include="number").columns:
    if df[column].isnull().any():
        median_value = df[column].median()
        if pd.notna(median_value):
            df[column] = df[column].fillna(median_value)

# STEP 9: Fill missing text values with the mode
for column in df.select_dtypes(include="object").columns:
    if df[column].isnull().any():
        mode_values = df[column].mode(dropna=True)
        if not mode_values.empty:
            df[column] = df[column].fillna(mode_values.iloc[0])

# STEP 10: Remove rows with invalid dates
if "date" in df.columns:
    df = df.dropna(subset=["date"]).copy()

# STEP 11: Remove invalid quantities and prices
if "quantity" in df.columns:
    df = df[df["quantity"] > 0].copy()

if "unit_price" in df.columns:
    df = df[df["unit_price"] >= 0].copy()

# STEP 12: Calculate sales
if {"quantity", "unit_price"}.issubset(df.columns):
    df["sales"] = df["quantity"] * df["unit_price"]

# STEP 13: Save the cleaned dataset
output_file = "sales_data_cleaned.csv"
df.to_csv(output_file, index=False)

# STEP 14: Verify the cleaned dataset
print("\n===== CLEANED DATASET =====")
print(df.head())

print("\nFinal dataset shape:", df.shape)
print("\nMissing values after cleaning:")
print(df.isnull().sum())

print("\nDuplicate rows after cleaning:", df.duplicated().sum())

print("\nTask 1 cleaning completed!")

# STEP 15: Download the cleaned CSV to your phone
files.download(output_file)


===== ORIGINAL DATASET =====
         Date   Product  Quantity  Unit_Price Region Payment_Method  \
0  2025-01-02    Laptop       2.0     50000.0  North           Card   
1  2025-01-03     Phone       1.0     20000.0  South            UPI   
2  2025-01-03     Phone       1.0     20000.0  South            UPI   
3  2025-01-05    Mouse        3.0       500.0   East           Cash   
4  2025-01-06  Keyboard       NaN      1200.0   West           Card   

   Customer_Age  
0          25.0  
1          31.0  
2          31.0  
3          22.0  
4          29.0  

Dataset shape: (12, 7)

Column names:
['Date', 'Product', 'Quantity', 'Unit_Price', 'Region', 'Payment_Method', 'Customer_Age']

Data types:
Date               object
Product            object
Quantity          float64
Unit_Price        float64
Region             object
Payment_Method     object
Customer_Age      float64
dtype: object

Missing values:
Date              0
Product           1
Quantity          1
Unit_Price        1
R

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>